# Structs, Records, and Data Models


```{contents}
:local:
:depth: 2
```


```{index} Choose Copy and Equality Semantics Together
```



```{index} Choose Copy and Equality Semantics Together
```

## Choose Copy and Equality Semantics Together



An order line refers to a product, but a point on a warehouse plan may simply consist of two coordinates. Before choosing class, struct or record, ask what one value means, whether it can change, and what callers should expect when it is assigned or compared. The forms below solve related modeling problems, but none is an automatic replacement for every ordinary class.

Assigning a struct variable copies its value. That copy contains the values of its fields; if one of those fields is itself a reference, both copies can still refer to a shared object. Value-type copying is not a promise that every reachable object has been recursively duplicated. A record class is a reference type with compiler-supported value-oriented equality and copying syntax; assigning its variable still copies a reference. A record struct combines record features with value-type copying.

| Choice | Assignment behavior | Design question |
| --- | --- | --- |
| Ordinary class instance | Another reference to the same object | Is identity and a controlled lifecycle important? |
| Struct value | Copy of its fields | Is this a small coherent value with a meaningful default? |
| Record class instance | Another reference on assignment | Does value-oriented equality fit a reference-based data model? |
| Record struct value | Copy of its fields | Do both value copying and record conveniences fit? |

For positional records, the parameter list declares the data used by generated members. A with expression creates a copied result with selected changes; it is not a deep-cloning operator. If a copied record contains a mutable list, later changes to that list may be visible through both record objects. Keep this distinction beside the syntax so that a concise expression does not suggest stronger isolation than it provides.

Default struct values deserve a separate check. The runtime can produce the zero-initialized value even when your preferred constructor would enforce a stronger business rule. If all-zero fields make no sense for the model, consider how every public operation handles that state rather than assuming clients can never obtain it. Readonly limits mutation through the struct's own fields; it does not freeze a separately referenced mutable collection.

Guided prediction: two variables hold equal coordinate values, then one is assigned a changed coordinate. Which variable changes? Now repeat with two references to one mutable class instance. Explain the observation using assignment semantics, not the spelling of the variable names. Finally, consider a record containing an array and identify which part of a with copy remains shared.

For related explanation, see the further reading below.[^development-reading]


```{index} Structs: Define a Value Type
```

(struct-value-types)=

## Structs: Define a Value Type




```{index} Structs: Define a Value Type
```


A delivery charge belongs to one quote. Copying that charge should give another value, rather than another reference to shared editable state. A **struct** lets you define a value type with named properties, constructors, and methods. Assignment and ordinary parameter passing copy its value, just as they do for `int` and `decimal`.[^1101_objects_references-structs]

Here `DeliveryCharge` groups an amount and a zone. `readonly` prevents its instance state from being changed after construction; the get-only properties expose its values. The explicit constructor uses the same syntax you learned for classes. Predict which amount changes when `revised` is assigned a new value.


In [1]:
using System;

DeliveryCharge original = new DeliveryCharge(5m, 2);
DeliveryCharge revised = original;
revised = new DeliveryCharge(7m, revised.Zone);
Console.WriteLine($"Original: {original.Amount:F2}, zone {original.Zone}");
Console.WriteLine($"Revised: {revised.Amount:F2}, zone {revised.Zone}");
Console.WriteLine($"Default amount: {default(DeliveryCharge).Amount:F2}");

readonly struct DeliveryCharge
{
    public decimal Amount { get; }
    public int Zone { get; }
    public DeliveryCharge(decimal amount, int zone)
    {
        if (amount < 0m || zone < 0) throw new ArgumentOutOfRangeException();
        Amount = amount;
        Zone = zone;
    }
}


Original: 5.00, zone 2
Revised: 7.00, zone 2
Default amount: 0.00


The two variables contain independent values. A mutable struct would also copy its fields on assignment, but mutable value types can make updates confusing, especially inside collections. Prefer a small, fixed value when introducing structs.

`default(DeliveryCharge)` supplies zero for both numeric properties without running this two-argument constructor. A struct design must account for its default value; constructor validation alone cannot exclude it. A non-nullable struct cannot contain `null`; `DeliveryCharge?` can, as the nullability reading explains.

Value copying is not a promise about stack versus heap storage. A struct can be a field inside a class or an element of an array. Also, if a struct contains a reference such as a `List<string>`, copying the struct copies that reference and can still share the list. Choose a type by its meaning and copying policy, not by assuming structs are always faster.


```{index} Records: Data and Equality
```

(record-data-types)=

## Records: Data and Equality




```{index} Records: Data and Equality
```


An invoice address is often compared by its contents. Two separately constructed addresses with the same city and postal code should compare equal even though they are different objects. A **record** asks the compiler to generate data-oriented members, including value-based equality and a readable `ToString`.[^1101_objects_references-records]

`record Address(string City, string Zip);` is short for a **record class** declaration. The parenthesized parameters generate a constructor and public properties with the same names. In this positional record class, those properties use `init`: they can be supplied during initialization but cannot be reassigned afterward. The following program uses ordinary construction, assignment, and equality before introducing copying with `with`.


In [2]:
using System;

Address first = new Address("Chicago", "60601");
Address alias = first;
Address imported = new Address("Chicago", "60601");
Address corrected = first with { Zip = "60602" };
Console.WriteLine($"Alias shares object: {ReferenceEquals(first, alias)}");
Console.WriteLine($"Imported shares object: {ReferenceEquals(first, imported)}");
Console.WriteLine($"Equal contents: {first == imported}");
Console.WriteLine($"Original ZIP: {first.Zip}");
Console.WriteLine($"Corrected ZIP: {corrected.Zip}");
Console.WriteLine(corrected);

record Address(string City, string Zip);


Alias shares object: True
Imported shares object: False
Equal contents: True
Original ZIP: 60601
Corrected ZIP: 60602
Address { City = Chicago, Zip = 60602 }


Assignment to `alias` still copies a reference: a record class remains a reference type. The generated `==` instead compares the record's data, so `first` and `imported` compare equal. The `with` expression creates a copy and initializes the listed changes; it leaves `first` unchanged. No mutation of the old address is needed.

Generated record equality is not automatically the business rule you need. Two product snapshots with the same SKU but different prices need not be equal as records. Decide whether the operation asks for matching SKU, matching all recorded data, or the identical object. An ordinary class does not automatically gain record-style `==` by having similar properties.

Records are not automatically deeply immutable. You can declare mutable properties, and even an init-only property can refer to a mutable list. Generated equality uses each member's equality behavior: distinct lists containing the same elements are not automatically compared element by element. The later copy-depth discussion applies to `with` too.


```{index} Record Structs and Choosing a Type
```

(record-struct-choice)=

## Record Structs and Choosing a Type




```{index} Record Structs and Choosing a Type
```


A **record struct** combines value-type copying with generated record members. Unlike a positional record class, a plain positional `record struct` generates settable properties. Use `readonly record struct` for a fixed value with init-only positional properties. Here each measurement is a small amount, not a shared entity.[^1101_objects_references-records]


In [3]:
using System;

PackageSize original = new PackageSize(20, 10);
PackageSize adjusted = original with { Width = 12 };
Console.WriteLine($"Original width: {original.Width}");
Console.WriteLine($"Adjusted width: {adjusted.Width}");
Console.WriteLine($"Equal values: {original == new PackageSize(20, 10)}");

readonly record struct PackageSize(int Length, int Width);


Original width: 10
Adjusted width: 12
Equal values: True


These declarations define one item's type. A collection such as `List<Address>` or `List<PackageSize>` holds multiple items; neither declaration replaces the collection.

| Declaration | Assignment | Equality to keep in mind | Typical role |
| --- | --- | --- | --- |
| Ordinary `class` | Copies a reference | Default `==` compares identity unless customized | An order with a lifecycle and shared state |
| `struct` | Copies its value | `Equals` compares fields by default; `==` is not automatically supplied | A small value with explicit members |
| `record` / `record class` | Copies a reference | Generated value-based equality | An address or snapshot compared by data |
| `record struct` | Copies its value | Generated value-based equality and `==` | A small data value with convenient record syntax |

For the class examples in this chapter, continue using ordinary classes. Choose a struct when independent value copying fits the domain; choose record syntax when its generated data comparison fits the intended rule. These choices do not remove validation responsibilities. In particular, positional records do not automatically check business constraints such as positive dimensions.

Before continuing, predict whether `Address b = a;` creates a new object and whether `PackageSize b = a;` can share mutable numeric properties. Then change one value in each runnable example and explain the output using copying and equality separately.


```{rubric} Footnotes
```


[^1101_objects_references-1]: A variable that stores a reference is separate from the object it refers to. Reference diagrams describe observable relationships; they do not promise a particular physical memory address or garbage-collector layout.
[^1101_objects_references-structs]: Microsoft, [Structure types](https://learn.microsoft.com/en-us/dotnet/csharp/language-reference/builtin-types/struct). See value copying, readonly structures, and default values.
[^1101_objects_references-records]: Microsoft, [Records](https://learn.microsoft.com/en-us/dotnet/csharp/language-reference/builtin-types/record). See positional properties, equality, and shallow copying with `with`.


[^development-reading]: Further reading: [Microsoft, C# Record Types](https://learn.microsoft.com/en-us/dotnet/csharp/fundamentals/types/records). The worked business scenarios and guided reasoning here are original teaching examples.
